# Explicit Midpoint Runge–Kutta Method

本 notebook 独立运行。对 Flow Matching ODE，Euler 只使用左端速度；Explicit Midpoint Runge–Kutta Method 先预测中点，再将中点速度用于整个区间。

$$k_1=v_\theta(x_n,t_n),$$

$$x_{mid}=x_n+\frac{h}{2}k_1,$$

$$k_2=v_\theta(x_{mid},t_n+\frac{h}{2}),$$

$$x_{n+1}=x_n+h k_2.$$

这是二阶方法：局部误差 O(h³)、全局误差 O(h²)，每步 2 NFE。第二次 decoder 的输入是预测半步状态 x_mid。


## 为什么中点一次预测就能达到二阶

精确积分中的速度平均值可以在真实时间中点取样：

$$
x_{n+1}=x_n+h f\left(x\left(t_n+\frac h2\right),t_n+\frac h2\right)+O(h^3).
$$

真中点状态未知，用左端 Euler 半步近似：

$$
x\left(t_n+\frac h2\right)=x_n+\frac h2 f_n+O(h^2).
$$

将其代入平滑的 $f$，中点速度误差仍是 $O(h^2)$；外面乘以 $h$ 后成为 $O(h^3)$ 的单步误差。因此更新

$$
x_{n+1}=x_n+h f\left(x_n+\frac h2 f_n,t_n+\frac h2\right)
$$

具有二阶全局精度。注意第二次网络调用不在真实轨迹上，而在预测的中点状态上；这就是 2 NFE 换取二阶精度的代价。


In [ ]:
import torch


class ExplicitMidpointRungeKuttaMethod:
    """Two-stage, second-order explicit Runge--Kutta midpoint method."""

    def __init__(self, velocity):
        self.velocity = velocity
        self.nfe = 0

    @torch.no_grad()
    def sample(self, x_init, times):
        x = x_init
        self.nfe = 0
        for t_now, t_next in zip(times[:-1], times[1:]):
            step = t_next - t_now
            k1 = self.velocity(x, t_now)  # Left-end velocity.
            x_mid = x + 0.5 * step * k1  # Euler prediction of x(t_n + h/2).
            k2 = self.velocity(x_mid, t_now + 0.5 * step)
            x = x + step * k2            # Midpoint quadrature of the integral.
            self.nfe += 2
        return x


times = torch.linspace(0.0, 1.0, 5)
solver = ExplicitMidpointRungeKuttaMethod(lambda x, t: torch.full_like(x, 2.0))
output = solver.sample(torch.zeros(1, 1), times)
torch.testing.assert_close(output, torch.full_like(output, 2.0))
assert solver.nfe == 8
print("constant-velocity invariant passed; NFE =", solver.nfe)
